# A1/A2 Data Inventory & EDA (CPU)

Menjalankan *source-file inventory* dan *exploratory data analysis* secara
reproduktif lewat logika `sipature_ml` — tanpa duplikasi bisnis-logic di notebook.
Ikuti `docs/eda-report.md`, `docs/data-inventory.md`, dan
`docs/reproducibility-runbook.md` sebelum eksekusi.

Notebook ini tidak membaca split/label apa pun; murni profiling data mentah.


## Step 1 — Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Konfigurasi path & parameter


In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DATASET_SOURCE_DIR = DRIVE_ROOT / "data" / "raw"  # raw CSV sumber di Drive

PROJECT_DIR = Path("/content/hackathon/ml")
LOCAL_DATASET_DIR = PROJECT_DIR / "data" / "raw"

REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"
FIGURE_DIR = PROJECT_DIR / "artifacts" / "figures" / "eda"

DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "eda"

SOURCE_ENCODING = "utf-8-sig"

print("Drive root:", DRIVE_ROOT)
print("Sumber dataset:", DATASET_SOURCE_DIR)
print("Dataset lokal:", LOCAL_DATASET_DIR)


Drive root: /content/drive/MyDrive/SIPATURE
Sumber dataset: /content/drive/MyDrive/SIPATURE/data/raw
Dataset lokal: /content/hackathon/ml/data/raw


## Step 3 — Clone repository dari GitHub


In [3]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

# GitHub menerima format Basic: x-access-token:<token>
credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



## Step 4 — Verifikasi commit terbaru (git log)


In [4]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
884175d (HEAD -> main, origin/main, origin/HEAD) feat: add reproducible data inventory and EDA notebook for Colab environments
722a53a chore: ignore .codebase-memory directory and remove .DS_Store from repository
1a2481e docs: expand SIPATURE analysis draft with model evaluation, comprehensive results summary, and technical roadmap for the final round.


## Step 5 — Install dependencies


In [5]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 699.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## Step 6 — Verifikasi versi package


In [3]:
import numpy
import pandas
import pyarrow
import sklearn
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)


NumPy: 2.2.6
Pandas: 2.3.3
PyArrow: 19.0.1
Scikit-learn: 1.7.2
Matplotlib: 3.10.3


## Step 7 — Copy dataset dari Drive ke lokal


In [4]:
# Salin raw CSV sumber dari Drive ke lokal agar sipature_ml membaca
# dari path yang deterministik.
import shutil
from pathlib import Path

LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_SOURCE_DIR.is_dir(), (
    f"Sumber dataset tidak ditemukan di Drive: {DATASET_SOURCE_DIR}\n"
    "Unggah CSV mentah ke folder tersebut sebelum melanjutkan."
)

copied = []
for source in sorted(DATASET_SOURCE_DIR.glob("*.csv")):
    destination = LOCAL_DATASET_DIR / source.name
    shutil.copy2(source, destination)
    copied.append(source.name)
    print("Disalin:", source.name)

print("\nTotal file:", len(copied))


Disalin: Dataset HackathonTourism - IT DEL.xlsx - Artikel Danau Toba.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - Attractions Info.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - Info Seputar Danau Toba (TOP 3).csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - hotel-metadata.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - hotel-resto-v1.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - kuliner.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - prompt.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - resto-hotel-v2.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - resto-metadata.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - tempat-wisata-v1.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - transportasi.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - waktu operasional destinasi.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - wisata-metadata.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - wisata-v2.csv

Total file: 14


## Step 8 — Import modul sipature_ml


In [5]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


## Step 9 — Load config & environment snapshot


In [6]:
from sipature_ml.config import load_config
from sipature_ml.environment import build_environment_snapshot

config = load_config("pipeline")
snapshot = build_environment_snapshot()

print("Pipeline version:", config["pipeline_version"])
print("Seed:", config["seed"])
print("Git commit:", snapshot["git_commit"])
print("Git dirty:", snapshot["git_dirty"])
print("Python:", snapshot["python"])


Pipeline version: 0.1.0
Seed: 42
Git commit: 884175d8161f779a4d0fbbbf4e70fd218c07695b
Git dirty: False
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## Step 10 — Jalankan data inventory


In [7]:
from sipature_ml.inventory import inventory_dataset, write_inventory

inventory = inventory_dataset(LOCAL_DATASET_DIR, encoding=SOURCE_ENCODING)

print("Dataset dir:", inventory["dataset_dir"])
print("File count:", inventory["file_count"])

json_path, csv_path = write_inventory(inventory, REPORT_DIR)
print("Inventory JSON:", json_path)
print("Inventory CSV :", csv_path)


Dataset dir: /content/hackathon/ml/data/raw
File count: 15
Inventory JSON: /content/hackathon/ml/artifacts/reports/data_inventory.json
Inventory CSV : /content/hackathon/ml/artifacts/reports/data_inventory.csv


## Step 11 — Lihat ringkasan inventory


In [8]:
# Tampilkan ringkasan inventory agar masalah terbaca langsung di notebook.
import pandas as pd

inventory_df = pd.DataFrame(inventory["files"])
inventory_df


,filename,size_bytes,sha256,suffix,columns,column_count,row_count_excluding_header,encoding,read_error
0,Dataset HackathonTourism - IT DEL.xlsx - Artik...,5865,70ab16901018d6c1e02e5118187b033ae0df48bc2dcd5c...,.csv,"[No, Judul Artikel, Kabupaten, Rangkuman (Para...",5.0,6.0,utf-8-sig,NaN
1,Dataset HackathonTourism - IT DEL.xlsx - Attra...,10153,ad29481ffa3377d2642ac48f9c7be3d7db9c0ebeb03ff1...,.csv,"[No, Nama Kabupaten, Nama Atraksi, Detail, , ,...",8.0,15.0,utf-8-sig,NaN
2,Dataset HackathonTourism - IT DEL.xlsx - Info ...,69200,dd9b285c48464d66108e6958e0c5c82f0360961b76a89d...,.csv,"[, , , , , , , , , , , , , , , , , , , , , , ,...",68.0,29.0,utf-8-sig,NaN
3,Dataset HackathonTourism - IT DEL.xlsx - hotel...,9620,e06375ebbeeea1c19654da7b5bc317bd2993eef534bdbd...,.csv,"[place-id, place-name, price-per-head, check-i...",12.0,36.0,utf-8-sig,NaN
4,Dataset HackathonTourism - IT DEL.xlsx - hotel...,3218,d2cfa6e35941560531aa1f31d6e14f588e2cb6c0a245dd...,.csv,"[place, rating, type, review, htm, facilitates...",8.0,9.0,utf-8-sig,NaN
5,Dataset HackathonTourism - IT DEL.xlsx - kulin...,6708,fd939c6fd5f66b60d74e4e61acee86366bfa51112773a5...,.csv,"[kuliner-id, kuliner-name, description]",3.0,12.0,utf-8-sig,NaN
6,Dataset HackathonTourism - IT DEL.xlsx - promp...,1729,3a1e24886fab02fbbd8c8f1482d7cd873e286de259d947...,.csv,"[, , ]",3.0,7.0,utf-8-sig,NaN
7,Dataset HackathonTourism - IT DEL.xlsx - resto...,1443999,f830a9b7b363207def7665c80436455b2f66deea23bf39...,.csv,"[place-name, reviewer-id, name, reviewer-ratin...",8.0,9611.0,utf-8-sig,NaN
8,Dataset HackathonTourism - IT DEL.xlsx - resto...,46141,b818a28ade9c600b8ea3ca91b71b5593c9126b25d59e07...,.csv,"[place-id, place-name, price-per-head, opening...",11.0,148.0,utf-8-sig,NaN
9,Dataset HackathonTourism - IT DEL.xlsx - tempa...,36703,f05f3d84a88ee786ea5b89c64cecd018d785d617f8cc48...,.csv,"[place, type, entry-fee, rating, addons, add, ...",9.0,96.0,utf-8-sig,NaN


## Step 12 — Jalankan EDA (generate figures)


In [9]:
from sipature_ml.eda import run_eda

eda_summary = run_eda(LOCAL_DATASET_DIR, REPORT_DIR, FIGURE_DIR)

print("Figures:", len(eda_summary["figures"]))
for name in eda_summary["figures"]:
    print("-", name)
print("\nReview summary keys:", sorted(eda_summary["review_summary"]))


Figures: 16
- 01_dataset_row_counts.png
- 02_review_availability_funnel.png
- 03_rating_distribution.png
- 04_review_text_length.png
- 05_top_place_review_coverage.png
- 06_place_text_coverage_bands.png
- 07_candidate_aspect_prevalence.png
- 08_language_negation_markers.png
- 09_file_missing_cell_rates.png
- 10_metadata_completeness_heatmap.png
- 11_metadata_coordinate_distribution.png
- 12_review_quality_anomalies.png
- 13_top_review_ngrams.png
- 14_review_time_field_availability.png
- 15_volume_vs_candidate_complaint_rate.png
- 16_nearby_service_density_5km.png

Review summary keys: ['blank_text_records', 'candidate_complaint_reviews', 'contrast_reviews', 'empty_review_records', 'exact_duplicate_excess_rows', 'integer_rating_distribution', 'language_marker_counts', 'negation_reviews', 'noninteger_rating_records', 'published_at_available', 'published_at_missing', 'rating_mean', 'rating_only_reviews', 'raw_rating_value_count', 'repeated_substantive_text_groups', 'repeated_text_excess_r

## Step 13 — Copy output ke Drive


In [10]:
# Salin output inventory + EDA ke Drive agar menjadi artefak persisten.
import shutil
from pathlib import Path

DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

report_files = [
    "data_inventory.json",
    "data_inventory.csv",
    "eda_summary.json",
    "eda_file_profile.csv",
    "eda_candidate_aspects.csv",
    "eda_place_coverage.csv",
    "eda_metadata_completeness.csv",
    "eda_coordinates.csv",
    "eda_ngrams.csv",
    "eda_service_density.csv",
]

for filename in report_files:
    source = REPORT_DIR / filename
    if source.is_file():
        shutil.copy2(source, DRIVE_REPORT_DIR / filename)
        print("Report disalin:", filename)

for name in eda_summary["figures"]:
    source = FIGURE_DIR / name
    if source.is_file():
        shutil.copy2(source, DRIVE_FIGURE_DIR / name)
        print("Figure disalin:", name)


Report disalin: data_inventory.json
Report disalin: data_inventory.csv
Report disalin: eda_summary.json
Report disalin: eda_file_profile.csv
Report disalin: eda_candidate_aspects.csv
Report disalin: eda_place_coverage.csv
Report disalin: eda_metadata_completeness.csv
Report disalin: eda_coordinates.csv
Report disalin: eda_ngrams.csv
Report disalin: eda_service_density.csv
Figure disalin: 01_dataset_row_counts.png
Figure disalin: 02_review_availability_funnel.png
Figure disalin: 03_rating_distribution.png
Figure disalin: 04_review_text_length.png
Figure disalin: 05_top_place_review_coverage.png
Figure disalin: 06_place_text_coverage_bands.png
Figure disalin: 07_candidate_aspect_prevalence.png
Figure disalin: 08_language_negation_markers.png
Figure disalin: 09_file_missing_cell_rates.png
Figure disalin: 10_metadata_completeness_heatmap.png
Figure disalin: 11_metadata_coordinate_distribution.png
Figure disalin: 12_review_quality_anomalies.png
Figure disalin: 13_top_review_ngrams.png
Figur

## Step 14 — Run summary (hash & limitations)


In [11]:
# ============================================================
# RUN SUMMARY — output path, hash sumber, dan limitations.
# ============================================================
import json
from sipature_ml.manifest import sha256_file

print("SOURCE HASHES:")
for name, digest in sorted(eda_summary["source_files"].items()):
    print(f"  {digest}  {name}")

print("\nOUTPUT REPORT DIR:", REPORT_DIR)
print("OUTPUT FIGURE DIR :", FIGURE_DIR)
print("DRIVE REPORT DIR  :", DRIVE_REPORT_DIR)
print("DRIVE FIGURE DIR  :", DRIVE_FIGURE_DIR)

print("\nEDA VERSION:", eda_summary["eda_version"])
print("LIMITATIONS:")
for item in eda_summary["limitations"]:
    print("  -", item)


SOURCE HASHES:
  70ab16901018d6c1e02e5118187b033ae0df48bc2dcd5cdb9211bcd1f4957905  Dataset HackathonTourism - IT DEL.xlsx - Artikel Danau Toba.csv
  ad29481ffa3377d2642ac48f9c7be3d7db9c0ebeb03ff17b6d03ee4589e56abe  Dataset HackathonTourism - IT DEL.xlsx - Attractions Info.csv
  dd9b285c48464d66108e6958e0c5c82f0360961b76a89d89374af1291967df9a  Dataset HackathonTourism - IT DEL.xlsx - Info Seputar Danau Toba (TOP 3).csv
  e06375ebbeeea1c19654da7b5bc317bd2993eef534bdbdb7da3a9a02226a16bd  Dataset HackathonTourism - IT DEL.xlsx - hotel-metadata.csv
  d2cfa6e35941560531aa1f31d6e14f588e2cb6c0a245ddce678e2b9d41516b1e  Dataset HackathonTourism - IT DEL.xlsx - hotel-resto-v1.csv
  fd939c6fd5f66b60d74e4e61acee86366bfa51112773a5ce2d9fba8df169dc9b  Dataset HackathonTourism - IT DEL.xlsx - kuliner.csv
  3a1e24886fab02fbbd8c8f1482d7cd873e286de259d9472a7f0e01792759677a  Dataset HackathonTourism - IT DEL.xlsx - prompt.csv
  f830a9b7b363207def7665c80436455b2f66deea23bf3930e3005bf2f73f0a1b  Dataset Hacka